In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import datetime
import csv

# Configuration
RESULTS_DIR = Path("../results")


In [ ]:
# Define time range for concatenating measurements
start_str = "2025-12-22 18:42:57"  # inclusive start
end_str = "2025-12-23 00:22:01"  

type_meas = "full_swing" # full_swing

fourier_analysis = True
plot_data = True


In [ ]:
%matplotlib widget

In [ ]:
start_dt = datetime.datetime.strptime(start_str, "%Y-%m-%d %H:%M:%S") - datetime.timedelta(minutes=1)
end_dt = datetime.datetime.strptime(end_str, "%Y-%m-%d %H:%M:%S") + datetime.timedelta(minutes=1)
if end_dt <= start_dt:
    raise ValueError("End datetime must be after start datetime")

# Find all files in the specified time range
files_in_range = []

current_date = start_dt.date()
while current_date <= end_dt.date():
    print(current_date)
    for file in sorted((RESULTS_DIR / current_date.strftime("%Y") / current_date.strftime("%m") / current_date.strftime("%d")).glob(f"{type_meas}_*.json")):
        # print(file.name)
        try:
            stamp = file.stem.split("_")[-1]  # expects v_meas_YYYYMMDD-HHMMSS.json
            dt = datetime.datetime.strptime(stamp, "%Y%m%d-%H%M%S")
            # print(dt)
        except ValueError:
            continue  # skip files that don't match the timestamp pattern
        if start_dt <= dt <= end_dt:
            print(dt)
            files_in_range.append((dt, file))
    current_date += datetime.timedelta(days=1)


In [ ]:

if not files_in_range:
    print("No files found in the specified range.")
else:
    # Load and concatenate data from all files
    concat_time = []
    concat_voltage_min = []
    concat_voltage_max = []
    concat_voltage_avg = []
    concat_voltage_diff = []
    base_dt = files_in_range[0][0]  # anchor absolute time to the first capture

    for dt, file in files_in_range:
        with open(file, "r") as f:
            d = json.load(f)
        t = np.array(d["time_s"], dtype=float)
        v = np.array(d["voltage_data_v"], dtype=float)
        if len(t) == 0 or len(v) == 0:
            continue

        # Offset this capture so its start reflects the true wall-clock interval
        offset = (dt - base_dt).total_seconds()
        concat_time.append([(t + offset)[0]])
        concat_voltage_min.append([np.min(v)])
        concat_voltage_max.append([np.max(v)])
        concat_voltage_avg.append([np.mean(v)])
        concat_voltage_diff.append([np.max(v) - np.min(v)])

    if not concat_time:
        print("All files in range were empty after parsing.")
    else:
        concat_time = np.concatenate(concat_time)
        concat_voltage_min = np.concatenate(concat_voltage_min)
        concat_voltage_max = np.concatenate(concat_voltage_max)
        concat_voltage_avg = np.concatenate(concat_voltage_avg)
        concat_voltage_diff = np.concatenate(concat_voltage_diff)
        
        print(f"Loaded {len(files_in_range)} files")
        print(f"Total data points: {len(concat_voltage_min)}")
        print(f"Time range: {base_dt} to {base_dt + datetime.timedelta(seconds=concat_time[-1])}")



In [ ]:
# Plot the concatenated data
if 'concat_time' in locals() and 'concat_voltage_avg' in locals() and len(concat_time) > 0:
    # Convert time to actual datetime objects starting from the first measurement time
    # base_dt is the datetime of the first measurement
    # concat_time is in seconds relative to base_dt, so add timedelta to base_dt
    from datetime import timedelta
    import matplotlib.dates as mdates

    
    concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time]

    if plot_data:

        fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
        # Plot min and max voltage on first axis
        axs[0].plot(concat_datetime[:], concat_voltage_avg[:], label='Avg Voltage', linewidth=1)
        axs[0].set_ylabel("Voltage (V)", fontsize=12)
        axs[0].legend()

        # Plot sum and diff voltage on second axis
        axs[1].plot(concat_datetime[:], concat_voltage_avg[:]/np.max(concat_voltage_avg[:]), label='Normalized Average Voltage', linewidth=1)
        axs[1].set_xlabel("Time", fontsize=12)
        axs[1].set_ylabel("Relative Power (alpha**2)", fontsize=12)
        axs[1].legend()

        # Format x-axis to show time nicely
        axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        axs[1].xaxis.set_major_locator(mdates.HourLocator(interval=1))
        plt.setp(axs[1].xaxis.get_majorticklabels(), rotation=45)
        axs[0].set_title(
            f"Absolute and relative average power: {start_str} to {end_str} \n {len(files_in_range)} files",
            fontsize=14,
        )
        for ax in axs:
            ax.grid(True, alpha=0.3)
        
        # Add text with normalized standard deviation
        std_normalized = np.std(concat_voltage_avg/np.max(concat_voltage_avg))
        axs[0].text(0.02, 0.98, f'Std(normalized): {std_normalized:.6f}', 
                    transform=axs[0].transAxes, fontsize=10, 
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.savefig(f"power_{start_str}_{end_str}.png")
        plt.show()

    print("\nConcatenated voltage stats:")
    print(f"  Std(avg): {np.std(concat_voltage_avg/np.max(concat_voltage_avg)):.6f} a.u.")
    print(f"  Avg(avg): {np.mean(concat_voltage_avg/np.max(concat_voltage_avg)):.6f} V")
    print("\nFiles included (chronological):")
    for dt, file in files_in_range:
        print(f"  {dt} -> {file.name}")
else:
    print("No data loaded. Please run the data loading cell first.")

In [ ]:
# Fourier analysis of normalized voltage
if 'concat_time' in locals() and 'concat_voltage_avg' in locals() and len(concat_time) > 0:
    # Normalize the voltage data
    normalized_voltage = concat_voltage_avg[:] / np.max(concat_voltage_avg[:])
    
    # Subtract the mean from the data
    demeaned_voltage = normalized_voltage - np.mean(normalized_voltage)
    
    # Apply Hamming window before FFT
    hamming_window = np.hamming(len(demeaned_voltage))
    windowed_voltage = demeaned_voltage * hamming_window
    
    # Calculate sampling rate from time data
    # Assuming time is in seconds and measurements are evenly spaced
    if len(concat_time) > 1:
        dt = np.mean(np.diff(concat_time))  # Average time step in seconds
        sampling_rate = 1.0 / dt  # Sampling rate in Hz
    else:
        sampling_rate = 1.0  # Default if only one point
        dt = 1.0
    
    # Perform FFT on windowed data
    fft_values = np.fft.fft(windowed_voltage)
    fft_magnitude = np.abs(fft_values)
    
    # Calculate frequency axis
    frequencies = np.fft.fftfreq(len(windowed_voltage), d=dt)
    
    # Only plot positive frequencies (up to Nyquist frequency), excluding DC component
    positive_freq_idx = (frequencies > 0) & (frequencies <= sampling_rate/2)
    frequencies_positive = frequencies[positive_freq_idx]
    fft_magnitude_positive = fft_magnitude[positive_freq_idx]
    
    # Convert frequencies to periods (in seconds)
    periods_positive = 1.0 / frequencies_positive
    
    # Sort by period (ascending) for better visualization
    sort_idx = np.argsort(periods_positive)
    periods_sorted = periods_positive[sort_idx]
    fft_magnitude_sorted = fft_magnitude_positive[sort_idx]
    
    # Plot Fourier spectrum in linear scale with period on x-axis (log scale)
    plt.figure(figsize=(12, 6))
    plt.plot(periods_sorted, fft_magnitude_sorted, linewidth=1)
    plt.xscale('log')
    plt.xlabel("Period (s)", fontsize=12)
    plt.ylabel("Magnitude (linear scale)", fontsize=12)
    plt.title(
        f"Fourier Transform of Normalized Voltage: {start_str} to {end_str}",
        fontsize=14,
    )
    plt.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(f"fourier_power_{start_str}_{end_str}.png")
    plt.show()
    
    # Print some statistics
    print("\nFourier Analysis Statistics:")
    print(f"  Sampling rate: {sampling_rate:.6f} Hz")
    print(f"  Nyquist frequency: {sampling_rate/2:.6f} Hz")
    print(f"  Number of points: {len(windowed_voltage)}")
    print(f"  Frequency resolution: {frequencies_positive[1] - frequencies_positive[0]:.6f} Hz")
    print(f"  Max magnitude: {np.max(fft_magnitude_positive):.6f}")
    max_mag_idx = np.argmax(fft_magnitude_positive)
    print(f"  Frequency at max magnitude: {frequencies_positive[max_mag_idx]:.6f} Hz")
    print(f"  Period at max magnitude: {periods_positive[max_mag_idx]:.6f} s")
else:
    print("No data loaded. Please run the data loading cell first.")


# Not Used

In [ ]:
# # Calculate and plot the phase
# if 'concat_time' in locals() and 'concat_voltage' in locals() and len(concat_time) > 0:
#     # Select the data range (same as used in phase function definition)
#     start_idx = 400  # Starting from 4 hours in (assuming 1000 samples per second)
#     time_selected = concat_time[start_idx:]
#     voltage_selected = concat_voltage[start_idx:]
    
#     # Calculate phase using the phase function
#     phase_shift = phase(voltage_selected, time_selected, v_min, v_max)
    
#     # Plot the phase
#     plt.figure(figsize=(12, 6))
#     plt.plot(time_selected, phase_shift, linewidth=0.5)
#     plt.xlabel("Time (s)", fontsize=12)
#     plt.ylabel("Phase (radians)", fontsize=12)
#     plt.title("Phase Shift vs Time", fontsize=14)
#     plt.grid(True, alpha=0.3)
#     plt.tight_layout()
#     plt.show()
    
#     # # Print phase statistics
#     # print("\nPhase statistics:")
#     # print(f"  Min phase: {np.min(phase_shift):.6f} rad ({np.min(phase_shift) * 180 / np.pi:.2f}°)")
#     # print(f"  Max phase: {np.max(phase_shift):.6f} rad ({np.max(phase_shift) * 180 / np.pi:.2f}°)")
#     # print(f"  Mean phase: {np.mean(phase_shift):.6f} rad ({np.mean(phase_shift) * 180 / np.pi:.2f}°)")
#     # print(f"  Phase range: {np.max(phase_shift) - np.min(phase_shift):.6f} rad ({(np.max(phase_shift) - np.min(phase_shift)) * 180 / np.pi:.2f}°)")
#     # print(f"  Number of points: {len(phase_shift)}")
# else:
#     print("No data loaded. Please run the data loading cell first.")

In [ ]:
# # Export concat_time and concat_voltage data
# if 'concat_time' in locals() and 'concat_voltage' in locals() and len(concat_time) > 0:
#     import csv
    
#     # Create filename with timestamp range
#     filename_base = f"concatenated_data_{start_str.replace(' ', '_').replace(':', '-')}_to_{end_str.replace(' ', '_').replace(':', '-')}"
    
#     # Option 1: Export to CSV (easy to open in Excel/spreadsheet programs)
#     export_filename_csv = f"{filename_base}.csv"
#     with open(export_filename_csv, 'w', newline='') as f:
#         writer = csv.writer(f)
#         writer.writerow(['time_s', 'voltage_v'])  # Header
#         # Write data in chunks to handle large files efficiently
#         chunk_size = 10000
#         for i in range(0, len(concat_time), chunk_size):
#             chunk = np.column_stack([concat_time[i:i+chunk_size], concat_voltage[i:i+chunk_size]])
#             writer.writerows(chunk)
#     print(f"Data exported to CSV: {export_filename_csv}")
#     print(f"  Rows: {len(concat_time)}")
    
#     # Option 2: Export to NPZ (numpy compressed format - preserves precision and is efficient)
#     export_filename_npz = f"{filename_base}.npz"
#     # Save base_dt as ISO format string for easy reconstruction
#     np.savez(export_filename_npz, 
#              concat_time=concat_time, 
#              concat_voltage=concat_voltage, 
#              base_dt_str=base_dt.isoformat(),
#              start_str=start_str,
#              end_str=end_str)
#     print(f"Data exported to NPZ: {export_filename_npz}")
#     print(f"  To load: data = np.load('{export_filename_npz}'); time = data['concat_time']; voltage = data['concat_voltage']")
#     print(f"  base_dt: {base_dt.isoformat()}")
# else:
#     print("No data loaded. Please run the data loading cell first.")

In [ ]:
# Exporting the data


In [ ]:
# v_min = 0.02489
# v_max = 0.0644

# v_min = min(v_min, np.min(concat_voltage[4*3600*1000:]))
# v_max = max(v_max, np.max(concat_voltage[4*3600*1000:]))


# def phase(v, t,v_min,v_max):
#     demeaned = v - (v_min + v_max)/2
#     swing = (v_max - v_min)/2
#     phase_shift = np.arcsin(demeaned/swing)
#     return phase_shift